In [4]:
#!/usr/bin/env python3
# =========================================================
# 🎯 HYBRID LEGAL SUMMARIZATION
# Role Importance (from training) × Neural Saliency (CNN-BiLSTM)
# NO TRAINING NEEDED - Direct application on test data
# =========================================================

import torch
import torch.nn as nn
import numpy as np
import json
import sys
import os

# Fix NLTK import issues - import before other packages
import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM,
    pipeline
)
import evaluate
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Using device: {device}")

# =========================================================
# MODEL LOADING
# =========================================================
print("\n📚 Loading models...")

# Legal LED for final summarization
led_model_name = "nsi319/legal-led-base-16384"
led_tokenizer = AutoTokenizer.from_pretrained(led_model_name)
led_model = AutoModelForSeq2SeqLM.from_pretrained(led_model_name).to(device)
led_model.eval()

# Rhetorical Role Classifier
role_model_name = "engineersaloni159/LegalRo-BERt_for_rhetorical_role_labeling"
role_pipe = pipeline(
    "text-classification",
    model=role_model_name,
    device=0 if torch.cuda.is_available() else -1,
    batch_size=32
)

# InLegalBERT for embeddings
embed_model_name = "law-ai/InLegalBERT"
embed_tokenizer = AutoTokenizer.from_pretrained(embed_model_name)
embed_model = AutoModel.from_pretrained(embed_model_name).to(device)
embed_model.eval()

print("✅ Models loaded successfully")

# =========================================================
# SENTENCE EMBEDDING
# =========================================================
@torch.no_grad()
def embed_sentences(sentences, batch_size=16):
    """Generate InLegalBERT embeddings."""
    all_embeddings = []
    
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        
        inputs = embed_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)

        outputs = embed_model(**inputs).last_hidden_state
        mask = inputs["attention_mask"].unsqueeze(-1)
        pooled = (outputs * mask).sum(1) / mask.sum(1)
        all_embeddings.append(pooled.cpu().numpy())
    
    return np.vstack(all_embeddings)

# =========================================================
# 🧠 CNN-BiLSTM FEATURE EXTRACTOR (UNSUPERVISED)
# =========================================================
class CNNBiLSTMFeatureExtractor(nn.Module):
    """
    Neural feature extractor for sentence saliency.
    
    NO TRAINING NEEDED - Uses architecture to extract features:
    1. CNN: Captures local legal phrase patterns
    2. BiLSTM: Captures document-level context
    3. Output: Rich sentence representation
    
    Saliency computed as similarity to document centroid.
    """
    def __init__(
        self, 
        embedding_dim=768,
        cnn_filters=512,
        cnn_kernel_size=3,
        lstm_hidden=32,
        lstm_layers=2,
        dropout=0.2
    ):
        super(CNNBiLSTMFeatureExtractor, self).__init__()
        
        # CNN: Detect legal phrase patterns
        self.conv1d = nn.Conv1d(
            in_channels=embedding_dim,
            out_channels=cnn_filters,
            kernel_size=cnn_kernel_size,
            padding=1
        )
        self.relu = nn.ReLU()
        
        # BiLSTM: Capture document context
        self.bilstm = nn.LSTM(
            input_size=cnn_filters,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0,
            bidirectional=True
        )
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, lengths=None):
        """
        Extract rich sentence features.
        
        Args:
            x: [batch, seq_len, embed_dim] - InLegalBERT embeddings
            lengths: Actual sequence lengths
            
        Returns:
            features: [batch, seq_len, lstm_hidden*2] - Rich representations
        """
        batch_size, seq_len, embed_dim = x.shape
        
        # CNN feature extraction
        x = x.transpose(1, 2)  # [batch, embed_dim, seq_len]
        x = self.conv1d(x)     # [batch, cnn_filters, seq_len]
        x = self.relu(x)
        x = x.transpose(1, 2)  # [batch, seq_len, cnn_filters]
        
        # BiLSTM context encoding
        if lengths is not None:
            x = nn.utils.rnn.pack_padded_sequence(
                x, lengths, batch_first=True, enforce_sorted=False
            )
        
        lstm_out, _ = self.bilstm(x)
        
        if lengths is not None:
            lstm_out, _ = nn.utils.rnn.pad_packed_sequence(
                lstm_out, batch_first=True
            )
        
        features = self.dropout(lstm_out)
        return features

# =========================================================
# INITIALIZE FEATURE EXTRACTOR (NO TRAINING)
# =========================================================
print("\n🔧 Initializing CNN-BiLSTM feature extractor...")
feature_extractor = CNNBiLSTMFeatureExtractor().to(device)
feature_extractor.eval()  # Evaluation mode (no training)
print("✅ Feature extractor ready (using random initialization)")

# =========================================================
# COMPUTE SENTENCE SALIENCY (UNSUPERVISED)
# =========================================================
@torch.no_grad()
def compute_sentence_saliency(sentences, embeddings):
    """
    Compute sentence-level saliency scores using CNN-BiLSTM features.
    
    Method: Centrality-based (similarity to document centroid)
    - Sentences similar to document center → High saliency
    - Peripheral sentences → Low saliency
    
    Args:
        sentences: List of sentence strings
        embeddings: InLegalBERT embeddings [N, 768]
        
    Returns:
        saliency_scores: [N] - Scores between 0 and 1
    """
    if len(sentences) == 0:
        return np.array([])
    
    # Prepare for CNN-BiLSTM
    embeddings_tensor = torch.FloatTensor(embeddings).unsqueeze(0).to(device)
    lengths = torch.LongTensor([len(sentences)])
    
    # Extract rich features through CNN-BiLSTM
    rich_features = feature_extractor(embeddings_tensor, lengths)
    rich_features = rich_features.squeeze(0).cpu().numpy()  # [N, lstm_hidden*2]
    
    # Compute document centroid
    centroid = rich_features.mean(axis=0, keepdims=True)
    
    # Saliency = Similarity to centroid (cosine similarity)
    similarities = cosine_similarity(rich_features, centroid).squeeze()
    
    # Normalize to [0, 1]
    if similarities.max() > similarities.min():
        saliency_scores = (similarities - similarities.min()) / (similarities.max() - similarities.min())
    else:
        saliency_scores = np.ones(len(sentences)) * 0.5
    
    # Additional factors (optional enhancements):
    # 1. Position bias: Earlier sentences slightly boosted
    position_weights = np.exp(-0.001 * np.arange(len(sentences)))
    
    # 2. Length penalty: Very short sentences penalized
    lengths_penalty = np.array([
        min(1.0, len(sent.split()) / 10.0) for sent in sentences
    ])
    
    # Combine factors
    saliency_scores = saliency_scores * position_weights * lengths_penalty
    
    # Re-normalize
    if saliency_scores.max() > 0:
        saliency_scores = saliency_scores / saliency_scores.max()
    
    return saliency_scores

# =========================================================
# 🎯 HYBRID SCORING: Role × Saliency
# =========================================================
def hybrid_sentence_selection(
    judgment,
    role_importance_path="importance_scores_improved.json",
    top_k=150,
    alpha=0.5,  # Weight for role importance
    beta=0.5    # Weight for saliency
):
    """
    Select sentences using HYBRID scoring:
    
    FinalScore(s_i) = α × RoleImportance(role_i) + β × SentenceSaliency(s_i)
    
    OR multiplicative:
    FinalScore(s_i) = RoleImportance(role_i) × SentenceSaliency(s_i)
    
    Steps:
    1. Classify sentences into rhetorical roles
    2. Get role importance from pre-computed JSON
    3. Compute sentence saliency using CNN-BiLSTM
    4. Combine scores
    5. Select top-K
    
    Args:
        judgment: Legal document text
        role_importance_path: Path to role_importance_scores.json
        top_k: Number of sentences to select
        alpha, beta: Weighting factors (for additive combination)
        
    Returns:
        selected_text: Concatenated top-K sentences
    """
    # Tokenize
    sentences = sent_tokenize(judgment)
    
    if len(sentences) == 0:
        return ""
    
    print(f"  📄 Document has {len(sentences)} sentences")
    
    # =====================================================
    # STEP 1: Classify Rhetorical Roles
    # =====================================================
    print("  🏷️  Classifying rhetorical roles...")
    role_preds = role_pipe(sentences, truncation=True)
    roles = [r["label"] for r in role_preds]
    
    # =====================================================
    # STEP 2: Load Role Importance Scores
    # =====================================================
    print("  📊 Loading role importance scores...")
    with open(role_importance_path, 'r') as f:
        importance_data = json.load(f)
    
    # Use hybrid scores (or any other scoring method from JSON)
    role_importance = importance_data['importance_scores']['hybrid_recommended']
    
    # Get importance for each sentence's role
    role_scores = np.array([
        role_importance.get(role, 0.01)  # Default to small value if role not found
        for role in roles
    ])
    
    print(f"  📈 Role score range: [{role_scores.min():.3f}, {role_scores.max():.3f}]")
    
    # =====================================================
    # STEP 3: Compute Sentence Saliency (CNN-BiLSTM)
    # =====================================================
    print("  🧠 Computing neural saliency scores...")
    embeddings = embed_sentences(sentences)
    saliency_scores = compute_sentence_saliency(sentences, embeddings)
    
    print(f"  📈 Saliency score range: [{saliency_scores.min():.3f}, {saliency_scores.max():.3f}]")
    
    # =====================================================
    # STEP 4: Combine Scores (HYBRID)
    # =====================================================
    print("  🔄 Combining role importance + neural saliency...")
    
    # METHOD A: Multiplicative (recommended - both factors must be high)
    final_scores = role_scores * saliency_scores
    
    # METHOD B: Additive (alternative - weighted sum)
    # final_scores = alpha * role_scores + beta * saliency_scores
    
    # METHOD C: Geometric mean (alternative - balanced)
    # final_scores = np.sqrt(role_scores * saliency_scores)
    
    print(f"  📈 Final score range: [{final_scores.min():.3f}, {final_scores.max():.3f}]")
    
    # =====================================================
    # STEP 5: Select Top-K Sentences
    # =====================================================
    top_k = min(top_k, len(sentences))  # Don't exceed available sentences
    top_indices = np.argsort(final_scores)[-top_k:]
    
    # Maintain document order
    selected_indices = sorted(top_indices)
    selected_sentences = [sentences[i] for i in selected_indices]
    
    print(f"  ✅ Selected {len(selected_sentences)} sentences")
    
    # =====================================================
    # OPTIONAL: Show score breakdown for top sentences
    # =====================================================
    print("\n  🔝 Top 5 Selected Sentences:")
    top_5_indices = np.argsort(final_scores)[-5:][::-1]
    for rank, idx in enumerate(top_5_indices, 1):
        print(f"    {rank}. Sentence {idx}")
        print(f"       Role: {roles[idx]:15s} (importance: {role_scores[idx]:.3f})")
        print(f"       Saliency: {saliency_scores[idx]:.3f}")
        print(f"       Final Score: {final_scores[idx]:.3f}")
        print(f"       Text: {sentences[idx][:80]}...")
    
    return " ".join(selected_sentences)

# =========================================================
# FINAL SUMMARY GENERATION (LED)
# =========================================================
def generate_summary(filtered_text, max_input_length=4096, max_output_length=512):
    """Generate abstractive summary using Legal LED."""
    inputs = led_tokenizer(
        filtered_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length
    ).to(device)

    global_attention_mask = torch.zeros_like(inputs["attention_mask"])
    global_attention_mask[:, 0] = 1

    with torch.no_grad():
        summary_ids = led_model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            global_attention_mask=global_attention_mask,
            max_length=max_output_length,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    return led_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# =========================================================
# EVALUATION
# =========================================================
def evaluate_model(preds, refs):
    """Evaluate using ROUGE and BERTScore."""
    print("\n📊 Computing ROUGE scores...")
    rouge = evaluate.load("rouge")
    rouge_scores = rouge.compute(predictions=preds, references=refs)
    
    print("📊 Computing BERTScore...")
    bertscore = evaluate.load("bertscore")
    bert_scores = bertscore.compute(
        predictions=preds,
        references=refs,
        model_type="roberta-large",
        lang="en"
    )

    results = {
        "ROUGE-1": rouge_scores["rouge1"],
        "ROUGE-2": rouge_scores["rouge2"],
        "ROUGE-L": rouge_scores["rougeL"],
        "BERTScore-F1": np.mean(bert_scores["f1"])
    }

    print("\n" + "="*60)
    print("📊 EVALUATION RESULTS")
    print("="*60)
    for metric, score in results.items():
        print(f"{metric:15s}: {score:.4f}")
    print("="*60)
    
    return results

# =========================================================
# 🎯 MAIN WORKFLOW
# =========================================================
if __name__ == "__main__":
    import argparse
    
    parser = argparse.ArgumentParser(
        description="Hybrid Legal Summarization: Role Importance × Neural Saliency"
    )
    parser.add_argument("--test_json", default="test.json", help="Path to test data")
    parser.add_argument("--role_importance", default="importance_scores_improved.json", 
                        help="Path to role importance JSON")
    parser.add_argument("--top_k", type=int, default=150, help="Number of sentences")
    parser.add_argument("--alpha", type=float, default=0.5, help="Weight for role importance")
    parser.add_argument("--beta", type=float, default=0.5, help="Weight for saliency")
    parser.add_argument("--output", default="summary_results.json", help="Output file")
    
    args = parser.parse_args()
    
    print("\n" + "="*60)
    print("🏛️  HYBRID LEGAL DOCUMENT SUMMARIZATION")
    print("="*60)
    print(f"Method: Role Importance × Neural Saliency (CNN-BiLSTM)")
    print(f"Test data: {args.test_json}")
    print(f"Role importance: {args.role_importance}")
    print(f"Top-K: {args.top_k}")
    print("="*60)
    
    # Check if role importance file exists
    if not os.path.exists(args.role_importance):
        print(f"\n❌ Error: Role importance file not found: {args.role_importance}")
        print("\nPlease ensure you have the role importance scores from training phase.")
        print("Expected format: JSON with 'importance_scores' -> 'hybrid_recommended' dict")
        sys.exit(1)
    
    # Load test data
    print(f"\n📂 Loading test data...")
    with open(args.test_json) as f:
        test_data = json.load(f)
    print(f"✅ Loaded {len(test_data)} test documents")
    
    # Generate summaries
    print(f"\n🔮 Generating summaries using hybrid approach...\n")
    predictions = []
    references = []
    
    for i, item in enumerate(tqdm(test_data, desc="Processing documents")):
        print(f"\n{'='*60}")
        print(f"Document {i+1}/{len(test_data)}")
        print(f"{'='*60}")
        
        # Hybrid sentence selection
        filtered_text = hybrid_sentence_selection(
            item["judgment"],
            role_importance_path=args.role_importance,
            top_k=args.top_k,
            alpha=args.alpha,
            beta=args.beta
        )
        
        # Generate final summary
        print("  📝 Generating LED summary...")
        summary = generate_summary(filtered_text)
        
        predictions.append(summary)
        references.append(item["reference_summary"])
        
        print(f"  ✅ Summary generated ({len(summary.split())} words)")
    
    # Evaluate
    print("\n" + "="*60)
    print("📊 EVALUATING RESULTS")
    print("="*60)
    results = evaluate_model(predictions, references)
    
    # Save results
    output_data = {
        "predictions": predictions,
        "references": references,
        "metrics": results,
        "config": {
            "method": "hybrid_role_saliency",
            "top_k": args.top_k,
            "alpha": args.alpha,
            "beta": args.beta,
            "role_importance_file": args.role_importance
        }
    }
    
    with open(args.output, "w") as f:
        json.dump(output_data, f, indent=2)
    
    print(f"\n💾 Results saved to: {args.output}")
    print("\n✅ PIPELINE COMPLETE!")

AttributeError: partially initialized module 'nltk' has no attribute 'data' (most likely due to a circular import)

In [2]:
# =========================================================
# 🎯 HYBRID LEGAL SUMMARIZATION (FINAL VERSION)
# Role Importance × Neural Saliency
# Fine-tuned LED model: "LED"
# Evaluation: ROUGE + BERTScore
# =========================================================

import torch
import torch.nn as nn
import numpy as np
import json
import os
import nltk
from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM,
    pipeline
)
import evaluate
from tqdm import tqdm

# =========================================================
# NLTK SETUP
# =========================================================
nltk.download("punkt")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("🖥️ Using device:", device)

# =========================================================
# LOAD MODELS
# =========================================================
print("\n📚 Loading models...")

# ✅ Your fine-tuned LED model
led_model_name = "LED"   # <---- your local fine-tuned model folder
led_tokenizer = AutoTokenizer.from_pretrained(led_model_name)
led_model = AutoModelForSeq2SeqLM.from_pretrained(led_model_name).to(device)
led_model.eval()

# Rhetorical Role Classifier
role_model_name = "engineersaloni159/LegalRo-BERt_for_rhetorical_role_labeling"
role_pipe = pipeline(
    "text-classification",
    model=role_model_name,
    device=0 if torch.cuda.is_available() else -1
)

# InLegalBERT for embeddings
embed_model_name = "law-ai/InLegalBERT"
embed_tokenizer = AutoTokenizer.from_pretrained(embed_model_name)
embed_model = AutoModel.from_pretrained(embed_model_name).to(device)
embed_model.eval()

print("✅ Models loaded successfully")

# =========================================================
# SENTENCE EMBEDDING
# =========================================================
@torch.no_grad()
def embed_sentences(sentences):
    inputs = embed_tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    ).to(device)

    outputs = embed_model(**inputs).last_hidden_state
    mask = inputs["attention_mask"].unsqueeze(-1)
    pooled = (outputs * mask).sum(1) / mask.sum(1)

    return pooled.cpu().numpy()

# =========================================================
# CNN-BiLSTM FEATURE EXTRACTOR (UNTRAINED)
# =========================================================
class CNNBiLSTMFeatureExtractor(nn.Module):
    def __init__(self, embedding_dim=768):
        super().__init__()

        self.conv = nn.Conv1d(
            in_channels=embedding_dim,
            out_channels=256,
            kernel_size=3,
            padding=1
        )

        self.lstm = nn.LSTM(
            input_size=256,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):
        x = x.transpose(1,2)
        x = torch.relu(self.conv(x))
        x = x.transpose(1,2)
        output,_ = self.lstm(x)
        return output

feature_extractor = CNNBiLSTMFeatureExtractor().to(device)
feature_extractor.eval()

# =========================================================
# COMPUTE SALIENCY
# =========================================================
@torch.no_grad()
def compute_saliency(sentences, embeddings):

    if len(sentences) == 0:
        return np.array([])

    tensor = torch.FloatTensor(embeddings).unsqueeze(0).to(device)
    features = feature_extractor(tensor).squeeze(0).cpu().numpy()

    centroid = features.mean(axis=0, keepdims=True)
    sims = cosine_similarity(features, centroid).squeeze()

    # Normalize to [0,1]
    if sims.max() > sims.min():
        sims = (sims - sims.min()) / (sims.max() - sims.min())
    else:
        sims = np.ones(len(sentences)) * 0.5

    return sims

# =========================================================
# HYBRID SENTENCE SELECTION
# =========================================================
def hybrid_selection(judgment, role_importance, top_k=150):

    sentences = sent_tokenize(judgment)
    if len(sentences) == 0:
        return ""

    # Step 1: Role prediction
    role_preds = role_pipe(sentences)
    roles = [r["label"] for r in role_preds]

    role_scores = np.array([
        role_importance.get(role, 0.01)
        for role in roles
    ])

    # Step 2: Embeddings
    embeddings = embed_sentences(sentences)

    # Step 3: Neural saliency
    saliency_scores = compute_saliency(sentences, embeddings)

    # Step 4: Hybrid scoring (multiplicative)
    final_scores = role_scores * saliency_scores

    top_k = min(top_k, len(sentences))
    indices = np.argsort(final_scores)[-top_k:]
    indices = sorted(indices)

    selected = [sentences[i] for i in indices]

    return " ".join(selected)

# =========================================================
# GENERATE SUMMARY USING FINE-TUNED LED
# =========================================================
def generate_summary(text):

    inputs = led_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(device)

    global_attention_mask = torch.zeros_like(inputs["attention_mask"])
    global_attention_mask[:,0] = 1

    with torch.no_grad():
        ids = led_model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            global_attention_mask=global_attention_mask,
            max_length=512,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    return led_tokenizer.decode(ids[0], skip_special_tokens=True)

# =========================================================
# EVALUATION (ROUGE + BERTScore)
# =========================================================
def evaluate_model(predictions, references):

    print("\n📊 Computing ROUGE...")
    rouge = evaluate.load("rouge")
    rouge_scores = rouge.compute(
        predictions=predictions,
        references=references
    )

    print("📊 Computing BERTScore...")
    bertscore = evaluate.load("bertscore")
    bert_scores = bertscore.compute(
        predictions=predictions,
        references=references,
        model_type="roberta-large",
        lang="en",
        device=device
    )

    results = {
        "ROUGE-1": rouge_scores["rouge1"],
        "ROUGE-2": rouge_scores["rouge2"],
        "ROUGE-L": rouge_scores["rougeL"],
        "BERTScore-F1": float(np.mean(bert_scores["f1"]))
    }

    print("\n" + "="*60)
    print("📊 FINAL EVALUATION RESULTS")
    print("="*60)
    for k,v in results.items():
        print(f"{k:15s}: {v:.4f}")
    print("="*60)

    return results

# =========================================================
# MAIN EXECUTION
# =========================================================

# Load role importance scores
with open("RR_scores.json") as f:
    importance_data = json.load(f)

role_importance = importance_data["importance_scores"]["hybrid_recommended"]

# Load test data
with open("test.json") as f:
    test_data = json.load(f)

predictions = []
references = []

print("\n🚀 Generating summaries...\n")

for item in tqdm(test_data):
    filtered = hybrid_selection(
        item["judgment"],
        role_importance,
        top_k=150
    )

    summary = generate_summary(filtered)

    predictions.append(summary)
    references.append(item["reference_summary"])

# =========================================================
# EVALUATE
# =========================================================
results = evaluate_model(predictions, references)

# Save output
output = {
    "predictions": predictions,
    "references": references,
    "metrics": results
}

with open("hybrid_summary_results.json", "w") as f:
    json.dump(output, f, indent=2)

print("\n💾 Results saved to hybrid_summary_results.json")
print("\n✅ PIPELINE COMPLETE!")


[nltk_data] Downloading package punkt to /home/RSlab/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


🖥️ Using device: cuda

📚 Loading models...
✅ Models loaded successfully

🚀 Generating summaries...



 10%|████████                                                                        | 10/100 [12:18<1:43:10, 68.79s/it]/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
 11%|████████▊                                                                       | 11/100 [13:31<1:43:51, 70.02s/it]/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
 12%|█████████▌                                                                      | 12/100 [14:51<1:47:07, 73.04s/it]/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially o


📊 Computing ROUGE...
📊 Computing BERTScore...


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



📊 FINAL EVALUATION RESULTS
ROUGE-1        : 0.4843
ROUGE-2        : 0.2525
ROUGE-L        : 0.2670
BERTScore-F1   : 0.8538

💾 Results saved to hybrid_summary_results.json

✅ PIPELINE COMPLETE!
